# 2-3 Hour Tutorial: Data Analysis & Visualisation with World Bank Nutrition Data
## Formulating SMART Questions and Solving Them with Pandas & Matplotlib

### Target Audience:
Beginners who have completed the basic file I/O and intro to Pandas tutorials.

### Learning Objectives:
By the end of this 2-3 hour tutorial, you will be able to:
1. **Formulate SMART questions** for data analytics projects.
2. **Reshape and transform** wide-format datasets into long-format datasets using `.melt()`.
3. **Clean missing data** and handle null values strategically.
4. **Analyze historical trends and distributions** across countries.
5. **Build expressive multi-line plots and ranked bar charts** to present your findings visually.

---

## Part 1: Formulating SMART Questions in Data Science (30 mins)

Before writing code, a data scientist must understand *what* they are looking for. We use the **SMART** framework to turn vague business/research goals into concrete analytical tasks:

* **S**pecific: Clearly states what needs to be measured (e.g., specific country or indicator).
* **M**easurable: Can be quantified using the metrics available in the dataset.
* **A**chievable: Is possible to answer given the available dimensions and data columns.
* **R**elevant: Solves a real-world problem or fits the project context (e.g., tracking malnutrition to evaluate health policy).
* **T**ime-bound: Restricts the window of observation to specific years or periods.

### Our Dataset
We are working with the World Bank World Development Indicators (WDI) dataset tracking the **"Prevalence of stunting, height for age (% of children under 5)"**.

### Turning Vague Questions into SMART Questions

1. **Vague:** "How is child stunting changing around the world?"
   * **SMART Question 1:** "What is the annual trend in child stunting percentages for **Nepal and Bhutan** over the 20-year period from **2000 to 2020**?"

2. **Vague:** "Which countries have the worst stunting problems?"
   * **SMART Question 2:** "Which **top 10 countries** recorded the highest prevalence of stunting in the year **2015**, and how do they compare relative to each other?"

3. **Vague:** "Is stunting going down globally?"
   * **SMART Question 3:** "What was the statistical distribution and median improvement in stunting rates globally between **2005 and 2020**?"

---

## Part 2: Loading & Reshaping Wide Data (45 mins)

### 💡 Mini-Tutorial: The Wide-to-Long Tidy Transformation
This dataset is stored in a **wide format**, where each year (`1983`, `1984`, ..., `2024`) has its own column. While this layout is easy for humans to read, it makes filtering, grouping, and plotting very difficult in Pandas.

To make the data **tidy**, we need to reshape it into a **long format** where each row represents a single observation for a country in a single year.

We achieve this using `pd.melt()`:
```python
df_long = pd.melt(df, 
                  id_vars=['REF_AREA_LABEL', 'INDICATOR_LABEL'], 
                  value_vars=['2000', '2001', '2002'], 
                  var_name='Year', 
                  value_name='Stunting_Rate')
```

Let's load the data file `WB_WDI_SH_STA_STNT_ZS_WIDEF.csv` and inspect it first.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the dataset
df = pd.read_csv('WB_WDI_SH_STA_STNT_ZS_WIDEF.csv')
print("Initial Dataset Shape:", df.shape)
print("Columns:", df.columns.tolist()[:10], "... plus year columns.")

### 📝 Exercise 2.1: Reshaping Data with `.melt()`
Transform the dataframe so that year columns from `2000` through `2023` are collapsed into two columns: `'Year'` and `'Stunting_Rate'`. Retain `'REF_AREA_LABEL'` as the country locator variable.

In [ ]:
# Define the list of years to melt
years_list = [str(y) for y in range(2000, 2024)]

# --- YOUR CODE HERE ---

### 🔍 Solution 2.1

In [ ]:
df_long = pd.melt(
    df, 
    id_vars=['REF_AREA_LABEL'], 
    value_vars=years_list, 
    var_name='Year', 
    value_name='Stunting_Rate'
)

# Rename REF_AREA_LABEL to Country for convenience
df_long = df_long.rename(columns={'REF_AREA_LABEL': 'Country'})
# Convert Year to numerical format
df_long['Year'] = df_long['Year'].astype(int)

print("Reshaped Long Dataset Sample:")
print(df_long.head())
print("New Shape:", df_long.shape)

--- 
## Part 3: Solving SMART Question 1 - Historical Trend Lines (30 mins)

> **SMART Question 1:** What is the annual trend in child stunting percentages for **Nepal and Bhutan** over the 20-year period from **2000 to 2020**?

### 💡 Mini-Tutorial: Multiple Filtering & Line Plotting
To extract specific locations, use the `.isin()` method:
```python
df_subset = df_long[df_long['Country'].isin(['Nepal', 'Bhutan'])]
```
When plotting multiple countries, we can slice individual subsets and layer them on the same Matplotlib chart.

### 📝 Exercise 3.1: Filtering & Plotting Trends
1. Filter `df_long` to extract rows where the country is `'Nepal'` or `'Bhutan'` and the year window is between `2000` and `2020` inclusive.
2. Drop missing rows (`.dropna()`) from the resulting subset.
3. Plot their trajectories on a single line chart with clear colors, markers, and a legend layout.

In [ ]:
# --- YOUR CODE HERE ---

### 🔍 Solution 3.1

In [ ]:
# 1. Filtering by values and temporal window
target_countries = ['Nepal', 'Bhutan']
df_q1 = df_long[
    (df_long['Country'].isin(target_countries)) &
    (df_long['Year'] >= 2000) &
    (df_long['Year'] <= 2020)
].dropna()

# 2. Plotting the lines
plt.figure(figsize=(9, 5))

for country in target_countries:
    country_data = df_q1[df_q1['Country'] == country]
    plt.plot(country_data['Year'], country_data['Stunting_Rate'], marker='o', label=country, linewidth=2)

plt.title("Stunting Prevalence Trends (2000-2020)", fontsize=13, fontweight='bold')
plt.xlabel("Observation Year")
plt.ylabel("Stunting Prevalence (% of children under 5)")
plt.xticks(range(2000, 2021, 2))
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()
plt.show()

--- 
## Part 4: Solving SMART Question 2 - Horizontal Ranked Comparisons (35 mins)

> **SMART Question 2:** Which **top 10 countries** recorded the highest prevalence of stunting in the year **2015**, and how do they compare relative to each other?

### 💡 Mini-Tutorial: Ordering & Truncating DataFrames
Sorting values is crucial when showing rankings:
```python
df_sorted = df.sort_values(by='column_name', ascending=False)
top_10 = df_sorted.head(10)
```
When displaying geographical names, using a horizontal bar plot (`plt.barh`) keeps country labels readable without overlap.

### 📝 Exercise 4.1: Top 10 Ranking Chart
1. Isolate the dataset to observations only containing the year `2015`.
2. Drop missing rows.
3. Sort the rows to find the top 10 countries with the highest `Stunting_Rate` values.
4. Render a horizontal bar chart displaying these top 10 positions in ascending sequence from bottom to top (invert the axis if needed).

In [ ]:
# --- YOUR CODE HERE ---

### 🔍 Solution 4.1

In [ ]:
# 1. Filter for 2015
df_2015 = df_long[(df_long['Year'] == 2015)].dropna()

# 2. Find top 10
top10_2015 = df_2015.sort_values(by='Stunting_Rate', ascending=False).head(10)

# 3. Plotting horizontal bar chart
plt.figure(figsize=(9, 5))
# Sort values ascending for nice visualization from top to bottom
top10_sorted_plot = top10_2015.sort_values(by='Stunting_Rate', ascending=True)

plt.barh(top10_sorted_plot['Country'], top10_sorted_plot['Stunting_Rate'], color='darkred', edgecolor='black')
plt.title("Top 10 Countries by Stunting Prevalence Rate in 2015", fontsize=13, fontweight='bold')
plt.xlabel("Prevalence percentage (% of children under 5)")
plt.ylabel("Country")
plt.grid(axis='x', linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

--- 
## Part 5: Solving SMART Question 3 - Comparative Distributions (40 mins)

> **SMART Question 3:** What was the statistical distribution and median improvement in stunting rates globally between **2005 and 2020**?

### 💡 Mini-Tutorial: Distribution Overlay Comparison
Histograms are excellent tools for comparing overall populations across time. To map historical shifts, we can overlay two histograms on the same figure layout and use the transparency parameter `alpha` (a value between 0 and 1) so overlapping areas remain clear.
```python
plt.hist(dataset1, alpha=0.5, label='Early Year')
plt.hist(dataset2, alpha=0.5, label='Later Year')
```

### 📝 Exercise 5.1: Global Distribution Overlay
1. Create two distinct filtering frames from `df_long`: one representing `2005` and another representing `2020`. Drop missing elements.
2. Calculate and print the `.median()` of the stunting rate for both target years.
3. Draw their distributions on a single plot canvas using overlapping histograms with `alpha=0.6`.

In [ ]:
# --- YOUR CODE HERE ---

### 🔍 Solution 5.1

In [ ]:
# 1. Separate series definitions
rates_2005 = df_long[(df_long['Year'] == 2005)].dropna()['Stunting_Rate']
rates_2020 = df_long[(df_long['Year'] == 2020)].dropna()['Stunting_Rate']

# 2. Medians reporting
print(f"Global Median Stunting Rate in 2005: {rates_2005.median():.2f}%")
print(f"Global Median Stunting Rate in 2020: {rates_2020.median():.2f}%")

# 3. Overlaid distribution histograms plotting
plt.figure(figsize=(9, 5))
plt.hist(rates_2005, bins=15, alpha=0.6, label='Year 2005', color='crimson', edgecolor='black')
plt.hist(rates_2020, bins=15, alpha=0.6, label='Year 2020', color='teal', edgecolor='black')

plt.title("Global Shift in Stunting Rates Distribution (2005 vs 2020)", fontsize=13, fontweight='bold')
plt.xlabel("Stunting Prevalence % Range")
plt.ylabel("Count of Countries")
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()